In [86]:
# Import das bibliotecas que iremos utilizar no projeto
import pandas as pd 
import requests 
import schedule
import time
import utils
import os 
import importlib

importlib.reload(utils)

<module 'utils' from 'C:\\Users\\Aguin\\Desktop\\Projetos\\gorest\\utils.py'>

In [ ]:
# Gerando a função que irá consumir a API e criará o DataFrame 
def conexao(link):
    # Variáveis de construção
    url = link
    dados = []
    # Paginação do arquivo por Páginas
    page = 1 
    # Cabeçário de identificação agente
    headers = {
        "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    # Utilização do loop while para fazer a requisição até que todas as páginas sejam entregues 
    while True:
        params = {
            "page":page,
            "per_page":30
        }
        req = requests.get(url, headers=headers, params=params)
        # Condicional para requisição somente dos dados 
        if req.status_code == 200:
            resposta = req.json()
        else:
            print("Houve um problema na requisição. COD: ", req.status_code)
            break

        if not resposta:
            break 

        dados.extend(resposta)
        page += 1

    df = pd.DataFrame(dados)
    return df

In [88]:
# Função que irá extrair cada tabela que se refere aos seguintes endpoints
# - Users, Posts, Carts e todos.
# Vale ressaltar que vou criar um dicionário que receberá a função de conexão
# e o argumentos passado será o link das APIs.
def extracao():
    tabelas = {
        "users":conexao("https://gorest.co.in/public/v2/users"),
        "posts":conexao("https://gorest.co.in/public/v2/posts"),
        "comments":conexao("https://gorest.co.in/public/v2/comments"),
        "todos":conexao("https://gorest.co.in/public/v2/todos")
    }
    return tabelas 

In [89]:
# Tratamento da tabela user com remoção de duplicadas, valores nulos
# e criação das colunas "bin" e "sexo" através da função aplly e lambda
def tratamento_user(df):
    df = df.copy()

    schema = {
        "id":"int",
        "name":"str",
        "email":"str",
        "gender":"str",
        "status":"str"
    }

    df = utils.tipagem_dados(df, schema)

    df = df.drop_duplicates(subset="id")
    df = df.dropna(subset="id")
    
    df["bin"] = df["status"].apply(lambda x: 1 if x == "active" else 0)
    df["sexo"] = df["gender"].apply(lambda x: "M" if x == "male" else "F")

    return df

In [90]:
# Tratamento da tabela post com remoção de duplicadas e valores nulos
def tratamento_post(df):
    df = df.copy()

    schema = {
        "id":"int",
        "user_id":"str", # A coluna user_id é tratada como identificação, então se utilizar string
        "title":"str",   # o power bi não utilizará ela como coluna numérica, mas sim como categórica.
        "body":"str"
    }
    df = utils.tipagem_dados(df, schema)
    df = df.drop_duplicates(subset="id")
    df = df.dropna(subset="id")

    return df

In [91]:
# Tratamento da tabela comentarios com apenas remoção de nullos e duplicadas 
def tratamento_comment(df):
    df = df.copy()

    schema = {
        "id":"int",
        "post_id":"str",
        "name":"str",
        "email":"str",
        "body":"str"
    }

    df = utils.tipagem_dados(df, schema)
    df = df.drop_duplicates(subset="id")
    df = df.dropna(subset="id")

    return df

In [92]:
# Tratamento da tabela todos comremoção de duplicadas, valores nullos,
# renome da coluna status para qualifaction e a criação da colunas 
# status para receber um condicional referente aos valores da então 
# colunas qualification.
def tratamento_todos(df):
    df = df.copy()

    schema = {
        "id":"int",
        "user_id":"str",
        "title":"str",
        "due_on":"datetime",
        "status":"str"
    }

    df = utils.tipagem_dados(df, schema)
    df = df.drop_duplicates(subset="id")
    df = df.dropna(subset="id")
    
    df = df.rename(columns={"status":"qualification"})
    df["status"] = df["qualification"].apply(lambda x: "active" if x == "completed" else "inactive")

    return df 

In [93]:
# A função de pipeline final, fará a importação de cada função
# de tratamento acima e aplicará em suas respectivas tabelas
def pipeline_tratamento():
    tabelas = extracao()

    tabelas_tratadas = {
        "users":tratamento_user(tabelas["users"]),
        "posts":tratamento_post(tabelas["posts"]),
        "comments":tratamento_comment(tabelas["comments"]),
        "todos":tratamento_todos(tabelas["todos"])
    }

    return tabelas_tratadas

In [136]:
# Função de exportação de tabelas 
def exportacao(tabelas_tratadas, diretorio="dados_tratados"):
    os.makedirs(diretorio, exist_ok=True)

    for nome, dados in tabelas_tratadas.items():
        caminho = os.path.join(diretorio, f"{nome}.csv")
        dados.to_csv(caminho, index=False, encoding="UTF-8")
        print("Arquivos exportados com sucesso")

In [134]:
# função de pipeline que executará as demais funções
def pipeline_final():
    print("Iniciando o processo de piepline...")

    tabelas = pipeline_tratamento()
    exportacao(tabelas)

    print("Pipeline executado com sucesso")

In [138]:
exe = pipeline_final()
print(exe)

Iniciando o processo de piepline...
Arquivos exportados com sucesso
Arquivos exportados com sucesso
Arquivos exportados com sucesso
Arquivos exportados com sucesso
Pipeline executado com sucesso
None
